# Cleaned bar-chart generator

This version mirrors the structure of the original notebook, but rewrites the generation pipeline for **bar charts** instead of pie charts.


In [137]:
# from __future__ import annotations

import json
import math
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import altair as alt
import kaleido
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import random

In [138]:
# Project paths
PROJECT = Path("..").resolve()
DATA_RAW = PROJECT / "data" / "raw"
DATA_PROCESSED = PROJECT / "data" / "processed"
DATA_TRAIN = PROJECT / "data" / "train"
OUTPUTS = PROJECT / "outputs"
OUT_ROOT = OUTPUTS / "generated" / "bar"

CLEAR_OUTPUT = True

LIBRARIES = ["altair", "matplotlib", "plotly"]
SUBDIRS = ["images", "tables", "meta"]

# Update this if the reference Excel file lives somewhere else inside the project
REFERENCE_XLSX = PROJECT / "data" / "references" / "bar charts correct.xlsx"
TRAIN_CSV = DATA_TRAIN / "Warehouse_and_Retail_Sales.csv"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

## 1. Style schema from the Excel sheet

In [139]:
STYLE_SPEC = {
    "title_present": {
        "column": 1,
        "default": True,
        "codes": {0: False, 1: True},
    },
    "title_location": {
        "column": 2,
        "default": "center",
        "codes": {0: "none", 1: "center", 2: "left", 3: "right"},
    },
    "title_color": {
        "column": 3,
        "default": "black",
        "codes": {0: "black", 1: "darkblue", 2: "darkred", 3: "darkgreen", 4: "gray"},
    },
    "subtitle_present": {
        "column": 4,
        "default": False,
        "codes": {0: False, 1: True},
    },
    "orientation": {
        "column": 5,
        "default": "vertical",
        "codes": {0: "vertical", 1: "horizontal"},
    },
    "bar_count": {
        "column": 6,
        "default": 6,
        "codes": {0: 4, 1: 5, 2: 6, 3: 8, 4: 10},
    },
    "sort_order": {
        "column": 7,
        "default": "descending",
        "codes": {0: "descending", 1: "ascending", 2: "alphabetical", 3: "reverse_alphabetical", 4: "original"},
    },
    "palette_type": {
        "column": 8,
        "default": "tab10",
        "codes": {
            0: "single_blue",
            1: "tab10",
            2: "pastel",
            3: "viridis",
            4: "muted_multicolor",
            5: "dark_palette",
            6: "rainbow_ordered",
        },
    },
    "single_color_mode": {
        "column": 9,
        "default": False,
        "codes": {0: False, 1: True},
    },
    "background": {
        "column": 10,
        "default": "white",
        "codes": {0: "white", 1: "light_gray", 2: "dark_gray", 3: "light_color", 4: "transparent"},
    },
    "grid_present": {
        "column": 11,
        "default": True,
        "codes": {0: False, 1: True},
    },
    "grid_axis": {
        "column": 12,
        "default": "y",
        "codes": {0: "y", 1: "x", 2: "both"},
    },
    "legend_present": {
        "column": 13,
        "default": False,
        "codes": {0: False, 1: True},
    },
    "legend_orientation": {
        "column": 14,
        "default": "right",
        "codes": {0: "right", 1: "left", 2: "top", 3: "bottom"},
    },
    "legend_title_present": {
        "column": 15,
        "default": False,
        "codes": {0: False, 1: True},
    },
    "x_label_present": {
        "column": 16,
        "default": True,
        "codes": {0: False, 1: True},
    },
    "y_label_present": {
        "column": 17,
        "default": True,
        "codes": {0: False, 1: True},
    },
    "category_label_rotation": {
        "column": 18,
        "default": 0,
        "codes": {0: 0, 1: 30, 2: 45, 3: 60, 4: 90},
    },
    "bar_border_width": {
        "column": 19,
        "default": 0,
        "codes": {0: 0, 1: 1, 2: 2},
    },
    "bar_border_color": {
        "column": 20,
        "default": "none",
        "codes": {0: "none", 1: "white", 2: "black", 3: "gray"},
    },
    "value_labels_present": {
        "column": 21,
        "default": True,
        "codes": {0: False, 1: True},
    },
    "value_label_position": {
        "column": 22,
        "default": "outside_end",
        "codes": {0: "outside_end", 1: "inside_end", 2: "center"},
    },
    "value_label_color": {
        "column": 23,
        "default": "black",
        "codes": {0: "black", 1: "white", 2: "same_as_title"},
    },
    "value_format": {
        "column": 24,
        "default": "integer",
        "codes": {0: "integer", 1: "1dp", 2: "2dp", 3: "compact"},
    },
    "show_values_as_share": {
        "column": 25,
        "default": False,
        "codes": {0: False, 1: True},
    },
    "image_outline": {
        "column": 26,
        "default": False,
        "codes": {0: False, 1: True},
    },
    "chart_width": {
        "column": 27,
        "default": 520,
        "codes": {0: 420, 1: 520, 2: 680},
    },
    "chart_height": {
        "column": 28,
        "default": 360,
        "codes": {0: 300, 1: 360, 2: 440},
    },
}

def normalize_choice(value):
    if pd.isna(value):
        return value
    if isinstance(value, np.generic):
        value = value.item()
    return value

def decode_style_value(raw_value, spec: dict):
    raw_value = normalize_choice(raw_value)
    if pd.isna(raw_value):
        return spec["default"]

    codes = spec.get("codes", {})
    if raw_value in codes:
        return codes[raw_value]

    if isinstance(raw_value, str):
        raw_value = raw_value.strip()
        if raw_value == "":
            return spec["default"]
        reverse_codes = {str(v).lower(): v for v in codes.values()}
        if raw_value.lower() in reverse_codes:
            return reverse_codes[raw_value.lower()]
        return raw_value

    return raw_value

In [140]:
# generate unique id names
def new_chart_id(prefix: str = "bar") -> str:
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"

# generate unique metadata files
def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for subdir in SUBDIRS:
        for library in LIBRARIES:
            (out_root / subdir / library).mkdir(parents=True, exist_ok=True)

def read_training_data(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported training-data file type: {path.suffix}")

def weighted_choice(rng: np.random.Generator, stats: dict):
    values = stats["values"]
    probs = np.array(stats["probs"], dtype=float)
    probs = probs / probs.sum()
    idx = rng.choice(len(values), p=probs)
    return normalize_choice(values[idx])

def infer_text_columns(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        series = df[col]
        if series.dtype == "O" or str(series.dtype).startswith("string") or pd.api.types.is_categorical_dtype(series):
            cols.append(col)
    return cols

def infer_numeric_columns(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)
    return cols

def value_to_display_text(value: float, fmt: str = "integer") -> str:
    value = float(value)
    if fmt == "compact":
        abs_value = abs(value)
        if abs_value >= 1_000_000_000:
            return f"{value/1_000_000_000:.1f}B"
        if abs_value >= 1_000_000:
            return f"{value/1_000_000:.1f}M"
        if abs_value >= 1_000:
            return f"{value/1_000:.1f}K"
        return f"{value:.0f}"
    if fmt == "2dp":
        return f"{value:.2f}"
    if fmt == "1dp":
        return f"{value:.1f}"
    return f"{value:.0f}"

In [141]:
def extract_reference_rows(excel_path: Path) -> pd.DataFrame:
    raw = pd.read_excel(excel_path, header=None)
    chart_mask = raw[0].astype(str).str.startswith("bar_img", na=False)
    return raw.loc[chart_mask].copy().reset_index(drop=True)

def parse_reference_styles(reference_rows: pd.DataFrame) -> pd.DataFrame:
    parsed = []
    for _, row in reference_rows.iterrows():
        style = {}
        for key, spec in STYLE_SPEC.items():
            col_idx = spec["column"]
            raw_value = row[col_idx] if col_idx < len(row) else np.nan
            style[key] = decode_style_value(raw_value, spec)
        parsed.append(style)
    return pd.DataFrame(parsed)

def compute_param_stats(reference_styles: pd.DataFrame) -> dict:
    stats = {}
    for key, spec in STYLE_SPEC.items():
        if reference_styles.empty:
            values = list(spec["codes"].values()) if spec.get("codes") else [spec["default"]]
            probs = np.repeat(1 / len(values), len(values))
        else:
            counts = reference_styles[key].value_counts(dropna=False)
            values = counts.index.tolist()
            probs = (counts / counts.sum()).tolist()
        stats[key] = {"values": values, "probs": probs}
    return stats

def load_reference_styles(excel_path: Path) -> tuple[pd.DataFrame, dict]:
    reference_rows = extract_reference_rows(excel_path)
    reference_styles = parse_reference_styles(reference_rows)
    param_stats = compute_param_stats(reference_styles)
    return reference_styles, param_stats

## 2. Style sampling and data sampling

In [142]:
def sample_bar_style(rng: np.random.Generator, param_stats: dict) -> dict:
    style = {}
    for key in STYLE_SPEC:
        style[key] = weighted_choice(rng, param_stats[key])

    style["bar_count"] = int(style["bar_count"])
    style["category_label_rotation"] = int(style["category_label_rotation"])
    style["chart_width"] = int(style["chart_width"])
    style["chart_height"] = int(style["chart_height"])
    style["bar_border_width"] = int(style["bar_border_width"])

    if style["single_color_mode"]:
        style["legend_present"] = False

    return style

def apply_sort_order(plot_df: pd.DataFrame, category_col: str, value_col: str, style: dict) -> pd.DataFrame:
    sort_order = style.get("sort_order", "descending")

    if sort_order == "descending":
        return plot_df.sort_values(value_col, ascending=False).reset_index(drop=True)
    if sort_order == "ascending":
        return plot_df.sort_values(value_col, ascending=True).reset_index(drop=True)
    if sort_order == "alphabetical":
        return plot_df.sort_values(category_col, ascending=True).reset_index(drop=True)
    if sort_order == "reverse_alphabetical":
        return plot_df.sort_values(category_col, ascending=False).reset_index(drop=True)

    return plot_df.reset_index(drop=True)

def sample_bar_data(df: pd.DataFrame, rng: np.random.Generator, style: dict) -> tuple[pd.DataFrame, dict] | None:
    category_candidates = infer_text_columns(df)
    numeric_candidates = infer_numeric_columns(df)

    if not category_candidates or not numeric_candidates:
        return None

    candidate_pairs = [(c, v) for c in category_candidates for v in numeric_candidates if c != v]
    rng.shuffle(candidate_pairs)

    for category_col, value_col in candidate_pairs:
        work = df[[category_col, value_col]].dropna().copy()
        if work.empty:
            continue

        work[category_col] = work[category_col].astype(str)
        work = work[work[category_col].str.strip() != ""]
        if work.empty:
            continue

        grouped = (
            work.groupby(category_col, dropna=False)[value_col]
            .sum()
            .reset_index()
        )
        grouped = grouped[grouped[value_col] > 0].reset_index(drop=True)
        if len(grouped) < 3:
            continue

        grouped = grouped.sort_values(value_col, ascending=False).reset_index(drop=True)
        n = min(style["bar_count"], len(grouped))
        if n < 3:
            continue

        if len(grouped) > n:
            top = grouped.iloc[:n].copy()
        else:
            top = grouped.copy()

        top = apply_sort_order(top, category_col, value_col, style)

        context = {
            "category_col": category_col,
            "value_col": value_col,
            "aggregation": "sum",
            "n_categories": int(len(top)),
            "orientation": style["orientation"],
        }
        return top, context

    return None

def make_title(context: dict) -> str:
    return f"{context['value_col']} by {context['category_col']}"

In [143]:
def build_value_label_column(plot_df: pd.DataFrame, value_col: str, style: dict) -> pd.Series:
    total = float(plot_df[value_col].sum())
    base = plot_df[value_col].astype(float)

    if style.get("show_values_as_share", False) and total > 0:
        return (100 * base / total).map(lambda x: f"{x:.0f}%")

    return base.map(lambda x: value_to_display_text(x, style.get("value_format", "integer")))

def compute_axis_limits(plot_df: pd.DataFrame, value_col: str, style: dict) -> tuple[float, float]:
    vmax = float(plot_df[value_col].max())
    if vmax <= 0:
        return 0.0, 1.0

    if style.get("value_labels_present", True) and style.get("value_label_position") == "outside_end":
        return 0.0, vmax * 1.18
    return 0.0, vmax * 1.08

def maybe_truncate_categories(plot_df: pd.DataFrame, category_col: str, max_len: int = 28) -> pd.DataFrame:
    df = plot_df.copy()
    df["_category_display"] = df[category_col].astype(str).map(
        lambda x: x if len(x) <= max_len else x[: max_len - 1] + "…"
    )
    return df

## 3. Shared styling helpers

In [144]:
def rgba_to_hex(rgba) -> str:
    return mcolors.to_hex(rgba, keep_alpha=False)

def get_background_color(style: dict) -> str:
    return {
        "transparent": "none",
        "white": "white",
        "light_gray": "#f1f3f5",
        "dark_gray": "#343a40",
        "light_color": "#f3f0ff",
    }.get(style.get("background", "white"), "white")

def get_title_fontsize(style: dict) -> int:
    width = int(style.get("chart_width", 520))
    if width <= 420:
        return 14
    if width >= 680:
        return 18
    return 16

def get_axis_text_color(style: dict) -> str:
    return "white" if style.get("background") == "dark_gray" else "black"

def get_grid_color(style: dict) -> str:
    return "#ced4da" if style.get("background") != "dark_gray" else "#adb5bd"

def get_plotly_title_anchor(style: dict) -> tuple[float, str]:
    location = style.get("title_location", "center")
    mapping = {
        "left": (0.0, "left"),
        "center": (0.5, "center"),
        "right": (1.0, "right"),
        "none": (0.5, "center"),
    }
    return mapping.get(location, (0.5, "center"))

def get_altair_title_anchor(style: dict) -> str:
    return {
        "left": "start",
        "center": "middle",
        "right": "end",
        "none": "middle",
    }.get(style.get("title_location", "center"), "middle")

def get_legend_title_text(category_col: str, style: dict) -> str | None:
    if style.get("legend_present") and style.get("legend_title_present"):
        return category_col
    return None

def build_palette(n: int, style: dict, rng: np.random.Generator | None = None) -> list[str]:
    palette_type = style.get("palette_type", "tab10")
    if style.get("single_color_mode", False):
        return ["#4c78a8"] * n

    if palette_type == "single_blue":
        return ["#4c78a8"] * n

    if palette_type == "pastel":
        cmap = plt.get_cmap("Pastel1")
        return [rgba_to_hex(cmap(i % 9)) for i in range(n)]

    if palette_type == "viridis":
        cmap = plt.get_cmap("viridis")
        return [rgba_to_hex(cmap(0.10 + 0.80 * i / max(1, n - 1))) for i in range(n)]

    if palette_type == "muted_multicolor":
        palette = ["#7b9acc", "#d4a373", "#84a98c", "#b5838d", "#9f86c0", "#8d99ae", "#6c9a8b"]
        return [palette[i % len(palette)] for i in range(n)]

    if palette_type == "rainbow_ordered":
        cmap = plt.get_cmap("rainbow")
        return [rgba_to_hex(cmap(i / max(1, n - 1))) for i in range(n)]

    if palette_type == "dark_palette":
        palette = ["#1b263b", "#415a77", "#5c677d", "#7d8597", "#9aa6b2", "#3a0ca3", "#560bad"]
        return [palette[i % len(palette)] for i in range(n)]

    return [rgba_to_hex(plt.get_cmap("tab10")(i % 10)) for i in range(n)]

def resolve_border_settings(style: dict) -> tuple[str | None, int]:
    if style.get("bar_border_width", 0) <= 0 or style.get("bar_border_color") == "none":
        return None, 0
    return style.get("bar_border_color", "black"), int(style.get("bar_border_width", 1))

def resolve_value_label_color(style: dict) -> str:
    color_mode = style.get("value_label_color", "black")
    if color_mode == "same_as_title":
        return style.get("title_color", "black")
    return color_mode

def add_subtitle_text(title: str, style: dict) -> str:
    if style.get("subtitle_present", False):
        return f"{title}\nGenerated chart"
    return title

def sanitize_style_common(plot_df: pd.DataFrame, style: dict) -> dict:
    style = dict(style)
    if len(plot_df) >= 9 and style.get("category_label_rotation", 0) == 0:
        style["category_label_rotation"] = 45
    if style.get("single_color_mode", False):
        style["legend_present"] = False
    return style

def add_value_labels_matplotlib(
    ax: plt.Axes,
    plot_df: pd.DataFrame,
    category_col: str,
    value_col: str,
    style: dict,
    orientation: str,
) -> None:
    if not style.get("value_labels_present", True):
        return

    labels = build_value_label_column(plot_df, value_col, style).tolist()
    color = resolve_value_label_color(style)
    pos = style.get("value_label_position", "outside_end")
    vmax = float(plot_df[value_col].max())

    for i, (value, label) in enumerate(zip(plot_df[value_col].tolist(), labels)):
        if orientation == "vertical":
            if pos == "inside_end":
                y = value - 0.04 * vmax
                va = "top"
            elif pos == "center":
                y = value / 2
                va = "center"
            else:
                y = value + 0.02 * vmax
                va = "bottom"
            ax.text(i, y, label, ha="center", va=va, color=color, fontsize=10)
        else:
            if pos == "inside_end":
                x = value - 0.02 * vmax
                ha = "right"
            elif pos == "center":
                x = value / 2
                ha = "center"
            else:
                x = value + 0.02 * vmax
                ha = "left"
            ax.text(x, i, label, ha=ha, va="center", color=color, fontsize=10)

def configure_matplotlib_grid(ax: plt.Axes, style: dict, orientation: str) -> None:
    if not style.get("grid_present", True):
        return
    axis_choice = style.get("grid_axis", "y")
    color = get_grid_color(style)
    if orientation == "vertical":
        if axis_choice in {"y", "both"}:
            ax.yaxis.grid(True, color=color, linewidth=0.8, alpha=0.7)
        if axis_choice in {"x", "both"}:
            ax.xaxis.grid(True, color=color, linewidth=0.8, alpha=0.35)
    else:
        if axis_choice in {"x", "both"}:
            ax.xaxis.grid(True, color=color, linewidth=0.8, alpha=0.7)
        if axis_choice in {"y", "both"}:
            ax.yaxis.grid(True, color=color, linewidth=0.8, alpha=0.35)
    ax.set_axisbelow(True)

## 4. Altair

In [145]:
def render_bar_altair(
    plot_df: pd.DataFrame,
    category_col: str,
    value_col: str,
    title: str,
    style: dict,
    rng: np.random.Generator | None = None,
) -> alt.Chart:
    plot_df = maybe_truncate_categories(plot_df, category_col)
    style = sanitize_style_common(plot_df, style)
    colors = build_palette(len(plot_df), style, rng=rng)
    plot_df["_color"] = colors
    plot_df["_value_label"] = build_value_label_column(plot_df, value_col, style)

    legend = alt.Legend(
        orient=style["legend_orientation"].replace("_", "-"),
        title=get_legend_title_text(category_col, style) or alt.Undefined,
        labelColor=get_axis_text_color(style),
        titleColor=get_axis_text_color(style),
    ) if style["legend_present"] else None

    title_kwargs = {}
    if style["title_present"]:
        title_kwargs["title"] = alt.TitleParams(
            text=title,
            fontSize=get_title_fontsize(style),
            color=style["title_color"],
            anchor=get_altair_title_anchor(style),
            subtitle=["Generated chart"] if style["subtitle_present"] else alt.Undefined,
        )

    x_axis = alt.Axis(
        title=category_col if style["x_label_present"] and style["orientation"] == "vertical" else (value_col if style["x_label_present"] and style["orientation"] == "horizontal" else None),
        labelAngle=style["category_label_rotation"] if style["orientation"] == "vertical" else 0,
        labelColor=get_axis_text_color(style),
        titleColor=get_axis_text_color(style),
        grid=bool(style["grid_present"] and style["grid_axis"] in {"x", "both"}),
    )
    y_axis = alt.Axis(
        title=value_col if style["y_label_present"] and style["orientation"] == "vertical" else (category_col if style["y_label_present"] and style["orientation"] == "horizontal" else None),
        labelColor=get_axis_text_color(style),
        titleColor=get_axis_text_color(style),
        grid=bool(style["grid_present"] and style["grid_axis"] in {"y", "both"}),
    )

    background_color = get_background_color(style)
    stroke_color, stroke_width = resolve_border_settings(style)

    if style["orientation"] == "vertical":
        base = alt.Chart(plot_df, **title_kwargs).properties(width=style["chart_width"], height=style["chart_height"])
        bars = base.mark_bar(stroke=stroke_color, strokeWidth=stroke_width).encode(
            x=alt.X("_category_display:N", sort=None, axis=x_axis),
            y=alt.Y(f"{value_col}:Q", scale=alt.Scale(domain=list(compute_axis_limits(plot_df, value_col, style))), axis=y_axis),
            color=alt.Color("_category_display:N", scale=alt.Scale(domain=plot_df["_category_display"].tolist(), range=colors), legend=legend),
            tooltip=[category_col, value_col],
        )
        layers = [bars]
        if style["value_labels_present"]:
            dy_map = {"outside_end": -8, "inside_end": 12, "center": 0}
            baseline_map = {"outside_end": "bottom", "inside_end": "top", "center": "middle"}
            layers.append(
                base.mark_text(
                    dy=dy_map[style["value_label_position"]],
                    color=resolve_value_label_color(style),
                    baseline=baseline_map[style["value_label_position"]],
                ).encode(
                    x=alt.X("_category_display:N", sort=None),
                    y=alt.Y(f"{value_col}:Q"),
                    text="_value_label:N",
                )
            )
    else:
        base = alt.Chart(plot_df, **title_kwargs).properties(width=style["chart_width"], height=style["chart_height"])
        bars = base.mark_bar(stroke=stroke_color, strokeWidth=stroke_width).encode(
            y=alt.Y("_category_display:N", sort=None, axis=y_axis),
            x=alt.X(f"{value_col}:Q", scale=alt.Scale(domain=list(compute_axis_limits(plot_df, value_col, style))), axis=x_axis),
            color=alt.Color("_category_display:N", scale=alt.Scale(domain=plot_df["_category_display"].tolist(), range=colors), legend=legend),
            tooltip=[category_col, value_col],
        )
        layers = [bars]
        if style["value_labels_present"]:
            dx_map = {"outside_end": 8, "inside_end": -8, "center": 0}
            align_map = {"outside_end": "left", "inside_end": "right", "center": "center"}
            layers.append(
                base.mark_text(
                    dx=dx_map[style["value_label_position"]],
                    color=resolve_value_label_color(style),
                    align=align_map[style["value_label_position"]],
                    baseline="middle",
                ).encode(
                    y=alt.Y("_category_display:N", sort=None),
                    x=alt.X(f"{value_col}:Q"),
                    text="_value_label:N",
                )
            )

    chart = alt.layer(*layers).configure_view(
        stroke="black" if style["image_outline"] else None,
        fill=None if background_color == "none" else background_color,
    ).configure_axis(
        domainColor=get_axis_text_color(style),
        tickColor=get_axis_text_color(style),
    )

    return chart

## 5. Matplotlib

In [146]:
def render_bar_matplotlib(
    plot_df: pd.DataFrame,
    category_col: str,
    value_col: str,
    title: str,
    style: dict,
    rng: np.random.Generator | None = None,
) -> plt.Figure:
    plot_df = maybe_truncate_categories(plot_df, category_col)
    style = sanitize_style_common(plot_df, style)
    colors = build_palette(len(plot_df), style, rng=rng)

    width = style.get("chart_width", 520)
    height = style.get("chart_height", 360)
    figsize = (width / 100, height / 100)

    background_color = get_background_color(style)
    facecolor = "none" if background_color == "none" else background_color
    fig, ax = plt.subplots(figsize=figsize, facecolor=facecolor)
    ax.set_facecolor(facecolor)

    border_color, border_width = resolve_border_settings(style)
    edgecolor = border_color if border_color is not None else "none"

    orientation = style["orientation"]
    if orientation == "vertical":
        ax.bar(
            plot_df["_category_display"],
            plot_df[value_col],
            color=colors,
            edgecolor=edgecolor,
            linewidth=border_width,
        )
        ymin, ymax = compute_axis_limits(plot_df, value_col, style)
        ax.set_ylim(ymin, ymax)
        ax.tick_params(axis="x", rotation=style["category_label_rotation"], colors=get_axis_text_color(style))
        ax.tick_params(axis="y", colors=get_axis_text_color(style))
        if style["x_label_present"]:
            ax.set_xlabel(category_col, color=get_axis_text_color(style))
        if style["y_label_present"]:
            ax.set_ylabel(value_col, color=get_axis_text_color(style))
    else:
        ax.barh(
            plot_df["_category_display"],
            plot_df[value_col],
            color=colors,
            edgecolor=edgecolor,
            linewidth=border_width,
        )
        xmin, xmax = compute_axis_limits(plot_df, value_col, style)
        ax.set_xlim(xmin, xmax)
        ax.tick_params(axis="x", colors=get_axis_text_color(style))
        ax.tick_params(axis="y", colors=get_axis_text_color(style))
        if style["x_label_present"]:
            ax.set_xlabel(value_col, color=get_axis_text_color(style))
        if style["y_label_present"]:
            ax.set_ylabel(category_col, color=get_axis_text_color(style))

    configure_matplotlib_grid(ax, style, orientation)
    add_value_labels_matplotlib(ax, plot_df, category_col, value_col, style, orientation)

    if style["title_present"]:
        loc = style.get("title_location", "center")
        if loc == "none":
            loc = "center"
        ax.set_title(
            add_subtitle_text(title, style),
            color=style["title_color"],
            fontsize=get_title_fontsize(style),
            loc=loc,
            pad=14,
        )

    if style["legend_present"] and not style.get("single_color_mode", False):
        handles = [
            plt.Line2D([0], [0], marker="s", color="none", markerfacecolor=color, markersize=8, label=cat)
            for cat, color in zip(plot_df["_category_display"].tolist(), colors)
        ]
        legend_title = get_legend_title_text(category_col, style)
        loc_map = {
            "right": "center left",
            "left": "center right",
            "top": "upper center",
            "bottom": "lower center",
        }
        bbox_map = {
            "right": (1.02, 0.5),
            "left": (-0.02, 0.5),
            "top": (0.5, 1.15),
            "bottom": (0.5, -0.20),
        }
        ax.legend(
            handles=handles,
            title=legend_title,
            loc=loc_map[style["legend_orientation"]],
            bbox_to_anchor=bbox_map[style["legend_orientation"]],
            frameon=False,
            labelcolor=get_axis_text_color(style),
        )

    if style.get("image_outline", False):
        fig.patch.set_edgecolor("black")
        fig.patch.set_linewidth(2.0)

    for spine in ax.spines.values():
        spine.set_color(get_axis_text_color(style))

    fig.tight_layout()
    return fig

## 6. Plotly

In [147]:
def get_plotly_legend_layout(style: dict) -> dict:
    orient = style.get("legend_orientation", "right")
    text_color = get_axis_text_color(style)
    layout = {
        "font": {"color": text_color},
        "title": {"text": "", "font": {"color": text_color, "size": max(10, get_title_fontsize(style) - 2)}},
        "bordercolor": "rgba(0,0,0,0)",
        "borderwidth": 0,
        "bgcolor": "rgba(0,0,0,0)",
    }

    if orient == "right":
        layout.update({"orientation": "v", "x": 1.02, "y": 0.5, "xanchor": "left", "yanchor": "middle"})
    elif orient == "left":
        layout.update({"orientation": "v", "x": -0.02, "y": 0.5, "xanchor": "right", "yanchor": "middle"})
    elif orient == "top":
        layout.update({"orientation": "h", "x": 0.5, "y": 1.10, "xanchor": "center", "yanchor": "bottom"})
    elif orient == "bottom":
        layout.update({"orientation": "h", "x": 0.5, "y": -0.14, "xanchor": "center", "yanchor": "top"})

    return layout

def render_bar_plotly(
    plot_df: pd.DataFrame,
    category_col: str,
    value_col: str,
    title: str,
    style: dict,
    rng: np.random.Generator | None = None,
):
    plot_df = maybe_truncate_categories(plot_df, category_col)
    style = sanitize_style_common(plot_df, style)
    colors = build_palette(len(plot_df), style, rng=rng)
    plot_df["_value_label"] = build_value_label_column(plot_df, value_col, style)

    title_x, title_anchor = get_plotly_title_anchor(style)
    background_color = get_background_color(style)
    border_color, border_width = resolve_border_settings(style)

    orientation = "v" if style["orientation"] == "vertical" else "h"

    fig = px.bar(
        plot_df,
        x="_category_display" if orientation == "v" else value_col,
        y=value_col if orientation == "v" else "_category_display",
        color="_category_display",
        color_discrete_sequence=colors,
        orientation=orientation,
        text="_value_label" if style["value_labels_present"] else None,
    )

    fig.update_traces(
        marker=dict(line=dict(color=border_color, width=border_width)),
        textposition={
            "outside_end": "outside",
            "inside_end": "inside",
            "center": "auto",
        }[style["value_label_position"]],
        textfont_color=resolve_value_label_color(style),
        cliponaxis=False,
        hovertemplate=f"{category_col}: %{{customdata[0]}}<br>{value_col}: %{{customdata[1]}}<extra></extra>",
        customdata=np.column_stack([plot_df[category_col], plot_df[value_col]]),
    )

    if style.get("single_color_mode", False):
        fig.update_traces(showlegend=False)
    else:
        fig.for_each_trace(lambda tr: tr.update(name=tr.name, showlegend=bool(style["legend_present"])))

    if orientation == "v":
        ymin, ymax = compute_axis_limits(plot_df, value_col, style)
        fig.update_yaxes(range=[ymin, ymax], showgrid=bool(style["grid_present"] and style["grid_axis"] in {"y", "both"}))
        fig.update_xaxes(tickangle=style["category_label_rotation"], showgrid=bool(style["grid_present"] and style["grid_axis"] in {"x", "both"}))
    else:
        xmin, xmax = compute_axis_limits(plot_df, value_col, style)
        fig.update_xaxes(range=[xmin, xmax], showgrid=bool(style["grid_present"] and style["grid_axis"] in {"x", "both"}))
        fig.update_yaxes(showgrid=bool(style["grid_present"] and style["grid_axis"] in {"y", "both"}))

    fig.update_layout(
        title=dict(
            text=add_subtitle_text(title, style) if style["title_present"] else "",
            x=title_x,
            xanchor=title_anchor,
            font=dict(size=get_title_fontsize(style), color=style.get("title_color", "black")),
        ),
        paper_bgcolor="rgba(0,0,0,0)" if background_color == "none" else background_color,
        plot_bgcolor="rgba(0,0,0,0)" if background_color == "none" else background_color,
        showlegend=bool(style["legend_present"] and not style.get("single_color_mode", False)),
        legend=get_plotly_legend_layout(style),
        margin=dict(l=50, r=40, t=90 if style["title_present"] else 40, b=60),
        width=style["chart_width"] + 80,
        height=style["chart_height"] + 60,
        font=dict(color=get_axis_text_color(style)),
    )

    legend_title_text = get_legend_title_text(category_col, style) or ""
    fig.update_layout(legend_title_text=legend_title_text)

    if not style["x_label_present"]:
        fig.update_xaxes(title=None)
    else:
        fig.update_xaxes(title=value_col if orientation == "h" else category_col)

    if not style["y_label_present"]:
        fig.update_yaxes(title=None)
    else:
        fig.update_yaxes(title=category_col if orientation == "h" else value_col)

    if style.get("image_outline", False):
        fig.update_layout(
            shapes=[
                dict(
                    type="rect",
                    xref="paper",
                    yref="paper",
                    x0=0,
                    y0=0,
                    x1=1,
                    y1=1,
                    line=dict(color="black", width=1.5),
                    fillcolor="rgba(0,0,0,0)",
                )
            ]
        )

    return fig

## 7. Saving, generation, and example usage

In [148]:
def save_altair_svg(chart, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    chart.save(str(out_path), format="svg")

def save_plotly_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out_path), format="png", scale=2)

def save_matplotlib_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        out_path,
        dpi=200,
        bbox_inches="tight",
        transparent=(fig.get_facecolor()[-1] == 0 if hasattr(fig.get_facecolor(), "__len__") else False),
    )
    plt.close(fig)

def generate_bar(
    df: pd.DataFrame,
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    param_stats: dict,
    max_tries: int = 50,
) -> dict:
    rng = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("bar")

    for attempt in range(1, max_tries + 1):
        style = sample_bar_style(rng, param_stats)
        sampled = sample_bar_data(df=df, rng=rng, style=style)
        if sampled is None:
            continue

        plot_df, context = sampled
        category_col = context["category_col"]
        value_col = context["value_col"]
        title = make_title(context)

        table_path = out_root / "tables" / library / f"{chart_id}.csv"
        meta_path = out_root / "meta" / library / f"{chart_id}.json"
        plot_df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.svg"
            chart = render_bar_altair(plot_df, category_col, value_col, title, style, rng=rng)
            save_altair_svg(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_matplotlib(plot_df, category_col, value_col, title, style, rng=rng)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_plotly(plot_df, category_col, value_col, title, style, rng=rng)
            save_plotly_png(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id": chart_id,
            "chart_type": "bar",
            "library": library,
            "dataset_source": dataset_source,
            "image_path": str(image_path),
            "table_path": str(table_path),
            "data_context": context,
            "style": style,
            "created_utc": datetime.utcnow().isoformat() + "Z",
            "attempt": attempt,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate a valid bar chart after {max_tries} attempts.")


def save_plot_df_table(plot_df: pd.DataFrame, table_path: Path):
    table_path.parent.mkdir(parents=True, exist_ok=True)
    plot_df.to_csv(table_path, index=False)

def generate_bar_from_plot_df(
    plot_df: pd.DataFrame,
    category_col: str,
    value_col: str,
    out_root: Path,
    chart_id: str,
    title: str,
    subtitle: str,
    library: str,
    style: dict,
    dataset_source: str = "test",
    attempt: int = 1,
) -> dict:
        
    ensure_output_dirs(out_root)

    table_path = out_root / "tables" / library / f"{chart_id}.csv"
    meta_path = out_root / "meta" / library / f"{chart_id}.json"
    plot_df.to_csv(table_path, index=False)

    if library == "altair":
        image_path = out_root / "images" / library / f"{chart_id}.svg"
        chart = render_bar_altair(
            plot_df, category_col, value_col, title, style
        )
        save_altair_svg(chart, image_path)

    elif library == "matplotlib":
        image_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_bar_matplotlib(
            plot_df, category_col, value_col, title, style
        )
        save_matplotlib_png(fig, image_path)

    elif library == "plotly":
        image_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_bar_plotly(
            plot_df, category_col, value_col, title, style
        )
        save_plotly_png(fig, image_path)

    else:
        raise ValueError(f"Unsupported library: {library}")

    context = {
        "category_col": category_col,
        "value_col": value_col,
        "subtitle": subtitle,
        "n_bars": len(plot_df),
    }

    meta = {
        "chart_id": chart_id,
        "chart_type": "bar",
        "library": library,
        "dataset_source": dataset_source,
        "image_path": str(image_path),
        "table_path": str(table_path),
        "data_context": context,
        "style": style,
        "created_utc": datetime.utcnow().isoformat() + "Z",
        "attempt": attempt,
    }

    save_metadata(meta, meta_path)
    return meta

def generate_batch(
    df: pd.DataFrame,
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    param_stats: dict,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed = start_seed

    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(
                generate_bar(
                    df=df,
                    out_root=out_root,
                    dataset_source=dataset_source,
                    library=library,
                    rng_seed=seed,
                    param_stats=param_stats,
                )
            )
            seed += 1

    return metas

In [149]:
# Optional run cell
# This only runs if both the training CSV and the Excel reference file exist at the paths above.

if TRAIN_CSV.exists() and REFERENCE_XLSX.exists():
    ensure_output_dirs(OUT_ROOT)

    df_train = read_training_data(TRAIN_CSV)
    reference_styles, PARAM_STATS = load_reference_styles(REFERENCE_XLSX)

    generation_plan = {
        "altair": 10,
        "matplotlib": 10,
        "plotly": 10,
    }

    metas = generate_batch(
        df=df_train,
        out_root=OUT_ROOT,
        dataset_source="Warehouse and Retail Sales",
        generation_plan=generation_plan,
        param_stats=PARAM_STATS,
        start_seed=1000,
    )

    pd.DataFrame(metas)[["chart_id", "library", "image_path"]].head()
else:
    print("Set TRAIN_CSV and REFERENCE_XLSX to your local files, then run this cell.")
    print("Missing TRAIN_CSV:", not TRAIN_CSV.exists())
    print("Missing REFERENCE_XLSX:", not REFERENCE_XLSX.exists())

Set TRAIN_CSV and REFERENCE_XLSX to your local files, then run this cell.
Missing TRAIN_CSV: False
Missing REFERENCE_XLSX: True


## 7. TEST RUN OF ALL PARAMETER OPTIONS INDIVIDUALLY

In [150]:
# ============================================================
# BAR TESTING EXPORT CELL
# - one chart per direct parameter setting
# - one chart per helper-behavior case
# - saves into: generated/bar_testing
# ============================================================

from pathlib import Path

TEST_OUT_ROOT = OUTPUTS / "generated" / "bar_testing"
TEST_LIBRARIES = ["altair", "matplotlib", "plotly"]

def default_param_stats_from_spec(style_spec: dict) -> dict:
    stats = {}
    for key, spec in style_spec.items():
        values = list(spec["codes"].values()) if spec.get("codes") else [spec["default"]]
        probs = np.repeat(1 / len(values), len(values))
        stats[key] = {"values": values, "probs": probs}
    return stats

def make_default_style() -> dict:
    return {key: spec["default"] for key, spec in STYLE_SPEC.items()}

def render_and_save_test_chart(
    plot_df: pd.DataFrame,
    category_col: str,
    value_col: str,
    title: str,
    style: dict,
    library: str,
    chart_id: str,
    out_root: Path,
) -> dict:
    table_path = out_root / "tables" / library / f"{chart_id}.csv"
    meta_path = out_root / "meta" / library / f"{chart_id}.json"
    table_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path.parent.mkdir(parents=True, exist_ok=True)
    plot_df.to_csv(table_path, index=False)

    if library == "altair":
        image_path = out_root / "images" / library / f"{chart_id}.svg"
        chart = render_bar_altair(plot_df, category_col, value_col, title, style, rng=np.random.default_rng(0))
        save_altair_svg(chart, image_path)
    elif library == "matplotlib":
        image_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_bar_matplotlib(plot_df, category_col, value_col, title, style, rng=np.random.default_rng(0))
        save_matplotlib_png(fig, image_path)
    elif library == "plotly":
        image_path = out_root / "images" / library / f"{chart_id}.png"
        fig = render_bar_plotly(plot_df, category_col, value_col, title, style, rng=np.random.default_rng(0))
        save_plotly_png(fig, image_path)
    else:
        raise ValueError(f"Unsupported library: {library}")

    meta = {
        "chart_id": chart_id,
        "chart_type": "bar",
        "library": library,
        "image_path": str(image_path),
        "table_path": str(table_path),
        "style": style,
        "created_utc": datetime.utcnow().isoformat() + "Z",
    }
    save_metadata(meta, meta_path)
    return meta

def build_test_dataset(
    df: pd.DataFrame,
    rng: random.Random | None = None,
    min_bars: int = 4,
    max_bars: int = 10,
):
    """
    Build a varied valid dataset for bar-chart testing.

    Variation dimensions:
    - different category/value column pairs
    - optional row subsampling
    - aggregation: sum / mean / median / count
    - number of bars shown
    - selection mode: top / bottom / random
    - ascending / descending sort
    """
    if df is None or df.empty:
        raise RuntimeError("Input dataframe is empty.")

    rng = rng or random.Random()

    work_df = df.copy()

    # Optional row subsampling for more variation
    if len(work_df) > 30 and rng.random() < 0.6:
        frac = rng.uniform(0.35, 0.9)
        work_df = work_df.sample(frac=frac, random_state=rng.randint(0, 10_000))

    numeric_cols = [
        c for c in work_df.columns
        if pd.api.types.is_numeric_dtype(work_df[c]) and work_df[c].notna().sum() > 0
    ]
    non_numeric_cols = [
        c for c in work_df.columns
        if not pd.api.types.is_numeric_dtype(work_df[c]) and work_df[c].notna().sum() > 0
    ]

    agg_options = ["sum", "mean", "median", "count"]
    select_modes = ["top", "bottom", "random"]
    sort_ascending = rng.choice([True, False])
    n_bars = rng.randint(min_bars, max_bars)

    candidate_pairs = []

    for category_col in non_numeric_cols:
        nunique = work_df[category_col].nunique(dropna=True)
        if 2 <= nunique <= 50:
            for value_col in numeric_cols:
                candidate_pairs.append((category_col, value_col))

    rng.shuffle(candidate_pairs)

    for category_col, value_col in candidate_pairs:
        agg = rng.choice(agg_options)
        tmp = work_df[[category_col, value_col]].dropna()

        if len(tmp) < min_bars:
            continue

        if agg == "count":
            grouped = (
                tmp.groupby(category_col, as_index=False)
                .size()
                .rename(columns={"size": "value"})
            )
            out_value_col = "value"
        else:
            grouped = (
                tmp.groupby(category_col, as_index=False)[value_col]
                .agg(agg)
            )
            out_value_col = value_col

        if len(grouped) < min_bars:
            continue

        # sort before selecting
        grouped = grouped.sort_values(out_value_col, ascending=sort_ascending)

        mode = rng.choice(select_modes)
        if len(grouped) > n_bars:
            if mode == "top":
                grouped = grouped.head(n_bars)
            elif mode == "bottom":
                grouped = grouped.tail(n_bars)
            else:
                grouped = grouped.sample(
                    n=n_bars,
                    random_state=rng.randint(0, 10_000)
                )

        # final sort for display
        if rng.random() < 0.7:
            grouped = grouped.sort_values(out_value_col, ascending=sort_ascending)
        else:
            grouped = grouped.sample(
                frac=1,
                random_state=rng.randint(0, 10_000)
            )

        grouped = grouped.reset_index(drop=True)
        return grouped, category_col, out_value_col

    # Final fallback: numeric-only data -> synthetic categories
    for value_col in numeric_cols:
        vals = work_df[value_col].dropna()
        if len(vals) >= min_bars:
            if len(vals) > n_bars:
                if rng.random() < 0.5:
                    vals = vals.sample(n=n_bars, random_state=rng.randint(0, 10_000))
                else:
                    vals = vals.head(n_bars)

            vals = vals.reset_index(drop=True)

            plot_df = pd.DataFrame({
                "category": [f"Item {i+1}" for i in range(len(vals))],
                value_col: vals.values,
            })
            return plot_df, "category", value_col

    raise RuntimeError("Could not build a valid aggregated test dataset for bar charts.")
def get_default_bar_style():
    return {
        # chart structure
        "orientation": "vertical",
        "sort_mode": "none",
        "top_n": None,
        "bargap": 0.2,

        # title / subtitle
        "title_present": True,
        "subtitle_present": False,
        "title_color": "#222222",

        # legend
        "legend_present": True,
        "legend_orientation": "right",

        # axis labels / ticks
        "x_label_present": True,
        "y_label_present": True,
        "category_label_rotation": 0,
        "tick_label_rotation": 0,
        "truncate_labels": False,
        "rotate_xticks": 0,

        # values on bars
        "show_values": False,
        "label_position": "end",

        # colors
        "color_mode": "single",
        "bar_color": "#4C78A8",
        "highlight_color": "#F58518",

        # axis / text styling
        "axis_text_color": "#333333",

        # grid
        "show_grid": True,
        "grid_present": True,
        "grid_axis": "y",   # "x", "y", or "both"

        # misc
        "background": "white",
    }

def run_direct_parameter_tests(df: pd.DataFrame, out_root: Path) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []

    direct_test_cases = [
        {"orientation": "vertical"},
        {"orientation": "horizontal"},
        {"show_values": True},
        {"show_values": False},
        {"show_grid": True},
        {"show_grid": False},
        {"sort_mode": "ascending"},
        {"sort_mode": "descending"},
        {"sort_mode": "none"},
        {"bargap": 0.05},
        {"bargap": 0.2},
        {"bargap": 0.4},
    ]

    lib_idx = 0

    for i, overrides in enumerate(direct_test_cases):
        rng = random.Random(1000 + i)

        try:
            plot_df, category_col, value_col = build_test_dataset(df, rng=rng)
        except RuntimeError:
            continue

        library = ["altair", "matplotlib", "plotly"][lib_idx % 3]
        lib_idx += 1

        style = get_default_bar_style().copy()
        style.update(overrides)

        title = f"Direct test {i+1}"
        subtitle = f"{category_col} vs {value_col}"

        chart_id = f"direct_test_{i+1:02d}_{library}"

        meta = generate_bar_from_plot_df(
            plot_df=plot_df,
            category_col=category_col,
            value_col=value_col,
            out_root=out_root,
            chart_id=chart_id,
            title=title,
            subtitle=subtitle,
            library=library,
            style=style,
            dataset_source="direct_parameter_test",
        )
        metas.append(meta)

    return metas
def run_helper_behavior_tests(df: pd.DataFrame, out_root: Path) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []

    helper_test_cases = [
        {"truncate_labels": True},
        {"truncate_labels": False},
        {"rotate_xticks": 0},
        {"rotate_xticks": 30},
        {"rotate_xticks": 60},
        {"label_position": "end"},
        {"label_position": "outside"},
        {"label_position": "none"},
        {"color_mode": "single"},
        {"color_mode": "category"},
        {"color_mode": "highlight_max"},
        {"top_n": 5},
        {"top_n": 8},
        {"top_n": 12},
    ]

    lib_idx = 0

    for i, overrides in enumerate(helper_test_cases):
        rng = random.Random(2000 + i)

        try:
            plot_df, category_col, value_col = build_test_dataset(df, rng=rng)
        except RuntimeError:
            continue

        library = ["altair", "matplotlib", "plotly"][lib_idx % 3]
        lib_idx += 1

        style = get_default_bar_style().copy()
        style.update(overrides)

        # If top_n exists, apply it to plot_df
        if "top_n" in style and len(plot_df) > style["top_n"]:
            top_n = style["top_n"]
            if value_col in plot_df.columns:
                plot_df = plot_df.sort_values(value_col, ascending=False).head(top_n).reset_index(drop=True)

        title = f"Helper test {i+1}"
        subtitle = f"{category_col} vs {value_col}"

        chart_id = f"helper_test_{i+1:02d}_{library}"

        meta = generate_bar_from_plot_df(
            plot_df=plot_df,
            category_col=category_col,
            value_col=value_col,
            out_root=out_root,
            chart_id=chart_id,
            title=title,
            subtitle=subtitle,
            library=library,
            style=style,
            dataset_source="helper_behavior_test",
        )
        metas.append(meta)

    return metas

if TRAIN_CSV.exists():
    df_train = read_training_data(TRAIN_CSV)
    direct_metas = run_direct_parameter_tests(df_train, TEST_OUT_ROOT)
    helper_metas = run_helper_behavior_tests(df_train, TEST_OUT_ROOT)

    test_summary = pd.DataFrame(direct_metas + helper_metas)
    print("Saved test charts:", len(test_summary))
    display(test_summary[["chart_id", "library", "image_path"]].head(20))
else:
    print("Missing TRAIN_CSV:", not TRAIN_CSV.exists())

KeyError: 'chart_width'